In [2]:
import chromadb
from datetime import datetime

#instantiate the client in memory
client = chromadb.Client()

#create a collection (unit of storage), if it doesn't exist already
#using the default embedding function
collection = client.create_collection(
    name="collection_one",
    metadata={
        "description": "my first Chroma collection",
        "created": str(datetime.now())
    }
    )

In [4]:
import pandas as pd

# Load the DataClaw conversations dataset from Hugging Face (https://huggingface.co/datasets/peteromallet/dataclaw-peteromallet)
df = pd.read_json("hf://datasets/peteromallet/dataclaw-peteromallet/conversations.jsonl", lines=True)

# Inspect the message keys
print(df["messages"][0][0].keys())
print(df["messages"][0][0])

dict_keys(['role', 'content', 'timestamp'])
{'role': 'user', 'content': 'List what directories and files are here. Just ls, no explanation needed.', 'timestamp': '2026-01-19T10:19:51.792Z'}


In [6]:
# Prepare documents, metadatas, and ids for insertion into Chroma
documents = []
metadatas = []
ids = []

# Iterate through the DataFrame and extract messages, along with metadata
for idx, row in df.iterrows():
    for msg_idx, msg in enumerate(row["messages"]):
        content = msg.get("content")
        if not content:
            continue
        # Use row index to guarantee uniqueness across duplicate session_ids
        doc_id = f"{idx}_{msg_idx}"
        documents.append(content)
        metadatas.append({
            "session_id": row["session_id"],
            "role": msg["role"],
            "project": str(row.get("project", "")),
            "model": str(row.get("model", "")),
        })
        ids.append(doc_id)

# Add in batches
batch_size = 5000
for i in range(0, len(documents), batch_size):
    collection.add(
        documents=documents[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size],
        ids=ids[i:i+batch_size],
    )

print(f"Added {len(documents)} documents to collection")

/Users/paradisdabbadon/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:00<00:00, 1.36MiB/s]


KeyboardInterrupt: 

The above cell was taking forever to complete (probably due to hardware bottlenecks). I was able to stop execution and still view a sizable portion of the data.